In [ ]:
# Week 2: Ingredient-based recipe recommendations
import ast
import re
from pathlib import Path
import pandas as pd
import numpy as np

df = pd.read_csv(Path("../data") / "cleaned_recipes.csv")
df.head()

In [ ]:
def parse_ingredients(raw):
    """Read original Food.com c("...", "...") ingredients safely."""
    if pd.isna(raw):
        return []
    text = str(raw).strip()
    if text.startswith("c(") and text.endswith(")"):
        text = text[2:-1].strip()
    if not text:
        return []
    try:
        parts = ast.literal_eval("[" + text + "]")
    except (ValueError, SyntaxError):
        parts = [part.strip().strip('"').strip("'") for part in text.split(",")]
    return [re.sub(r"\\s+", " ", str(item)).strip().lower()
            for item in parts if str(item).strip()]

# Always use the original RecipeIngredientParts, not the old cleaned_ingredients.
df["ingredient_list"] = df["RecipeIngredientParts"].apply(parse_ingredients)
df[["Name", "ingredient_list"]].head()

In [ ]:
# Change this list to ingredients available in the user's kitchen.
user_ingredients = {"blueberries", "sugar", "vanilla yogurt", "flour", "eggs"}

def normalize_pantry(ingredients):
    return {str(item).strip().lower() for item in ingredients if str(item).strip()}

def ingredient_match_score(recipe_ingredients, pantry):
    recipe = set(recipe_ingredients)
    return len(recipe & pantry) / len(recipe) if recipe else 0.0

def only_user_ingredients(recipe_ingredients, pantry):
    recipe = set(recipe_ingredients)
    return bool(recipe) and recipe.issubset(pantry)

In [ ]:
def recommend_recipes(recipes, pantry_ingredients, min_match=0.3, prior_reviews=20):
    """Return general and strict recommendations, recomputing scores each call.

    prior_reviews controls how strongly ratings with few reviews shrink
    toward the dataset's overall average rating.
    """
    pantry = normalize_pantry(pantry_ingredients)
    scored = recipes.copy()
    scored["match_score"] = scored["ingredient_list"].apply(
        lambda items: ingredient_match_score(items, pantry)
    )
    scored["missing_ingredients"] = scored["ingredient_list"].apply(
        lambda items: sorted(set(items) - pantry)
    )
    scored["is_strict_match"] = scored["ingredient_list"].apply(
        lambda items: only_user_ingredients(items, pantry)
    )

    # Stable Bayesian shrinkage: rating confidence grows with review count.
    ratings = pd.to_numeric(scored["AggregatedRating"], errors="coerce")
    counts = pd.to_numeric(scored["ReviewCount"], errors="coerce").fillna(0).clip(lower=0)
    baseline = ratings[(ratings >= 0) & (ratings <= 5)].mean()
    if pd.isna(baseline):
        baseline = 2.5
    valid_ratings = ratings.where(ratings.between(0, 5), baseline).fillna(baseline)
    counts = counts.where(ratings.between(0, 5), 0)
    scored["weighted_rating"] = (
        counts * valid_ratings + prior_reviews * baseline
    ) / (counts + prior_reviews)
    scored["final_score"] = (
        0.7 * scored["match_score"] + 0.3 * (scored["weighted_rating"] / 5)
    )

    general = scored.loc[scored["match_score"] >= min_match].copy()
    strict = general.loc[general["is_strict_match"]].copy()
    general = general.sort_values("final_score", ascending=False)
    strict = strict.sort_values("final_score", ascending=False)
    return general, strict

In [ ]:
# Re-run this cell whenever user_ingredients changes.
filtered, filtered_strict = recommend_recipes(df, user_ingredients)

print(f"General matches: {len(filtered):,}")
print(f"Strict pantry-only matches: {len(filtered_strict):,}")
print("Strict matches cannot exceed general matches.")
assert len(filtered_strict) <= len(filtered)

In [ ]:
# Show the best general recommendations, including what's missing.
columns = [
    "RecipeId", "Name", "AuthorName", "ingredient_list",
    "missing_ingredients", "match_score", "AggregatedRating",
    "ReviewCount", "weighted_rating", "final_score"
]
display(filtered[columns].head(25))

In [ ]:
# Only recipes requiring NO ingredients outside the user's pantry.
display(filtered_strict[columns].head(25))

In [ ]:
# Quick tests that don't require the Food.com dataset.
assert parse_ingredients('c("cinnamon", "chives")') == ["cinnamon", "chives"]
assert only_user_ingredients(["rice", "onion"], {"rice", "onion", "garlic"})
assert not only_user_ingredients(["rice", "milk"], {"rice", "onion"})
assert not only_user_ingredients([], {"rice"})
assert ingredient_match_score(["rice", "onion"], {"rice"}) == 0.5

test_df = pd.DataFrame({
    "RecipeId": [1, 2, 3],
    "Name": ["Rice", "Rice and milk", "Onion"],
    "AuthorName": ["A", "B", "C"],
    "ingredient_list": [["rice"], ["rice", "milk"], ["onion"]],
    "AggregatedRating": [5.0, 4.75, 4.0],
    "ReviewCount": [1, 1000, 50],
})
test_general, test_strict = recommend_recipes(test_df, {"rice", "onion"})
assert set(test_strict["RecipeId"]) == {1, 3}
assert set(test_general["RecipeId"]) == {1, 2, 3}
assert test_general.loc[test_general["RecipeId"] == 2, "missing_ingredients"].iloc[0] == ["milk"]
print("Parser, strict-filter, and scoring smoke tests passed.")